# ACE candidate drugability — ADME descriptors and Glaxo structural alerts (revised)

Merges the classifier and regressor screens, computes transparent RDKit descriptors, and applies the supplied `glaxo_structural_alerts.csv` SMARTS rules. This is a prioritisation filter, not an ADMET or safety determination.

**What changed vs. the previous version**
- Libraries contain repeated structures (e.g. several records per compound in NPASS/CMAUP), so the old `validate='one_to_one'` merge would raise a `MergeError`. Rows are now collapsed to one per `(source, canonical_smiles)` before merging.
- The regression cutoff now follows the classifier: 25 nM = pChEMBL 7.60 (both models must call a compound active at the same potency). Expect fewer candidates than with the old 1,000 nM cutoff.
- Descriptors and alerts are computed only for model-consensus candidates (the rest cannot pass anyway), which avoids running ~150k molecules through every SMARTS pattern.
- Fixed-name helpers from `ace_config` (`read_table`, `locate_column`) are no longer required; the alert table is read directly with column auto-detection.
- A positive-control check runs known ACE drugs through the same filters, because Lipinski/Veber limits and thiol/carboxylate alerts can penalise genuine ACE chemotypes.

In [1]:
from pathlib import Path
import os, sys, json, warnings
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path('/content/drive/MyDrive/ACE_Workflow')
except ImportError:
    ROOT = Path(os.environ.get('ACE_WORKFLOW_ROOT', Path.cwd())).resolve()
if not (ROOT / 'ace_config.py').exists():
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ace_config.py').exists()), ROOT)
if not (ROOT / 'ace_config.py').exists():
    raise FileNotFoundError('Could not find ace_config.py. Mount Drive or set ACE_WORKFLOW_ROOT.')
os.environ['ACE_WORKFLOW_ROOT'] = str(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import ace_config

import csv, numpy as np, pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Crippen, Descriptors, Lipinski, rdMolDescriptors
RDLogger.DisableLog('rdApp.*')

PATHS, INPUTS = ace_config.PATHS, ace_config.INPUTS
CLASSIFIER_CUTOFF = 0.50
# Regression cutoff equals the classifier's activity definition: 25 nM -> pChEMBL 7.60.
REGRESSION_CUTOFF_PCHEMBL = 9.0 - np.log10(ace_config.IC50_CUTOFF_NM)
MAX_GLAXO_ALERTS = 0            # raise only with a documented rationale
OUT = PATHS['DRUGABILITY']; OUT.mkdir(exist_ok=True, parents=True)
REPORTS = PATHS['DRUGABILITY_REPORTS']; REPORTS.mkdir(exist_ok=True, parents=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
classifier_file = PATHS['ML_CLASSIFICATION'] / 'all_libraries_ic50_screen.csv'
regressor_file = PATHS['ML_REGRESSION'] / 'all_libraries_pchembl_screen.csv'
if not classifier_file.exists() or not regressor_file.exists():
    raise FileNotFoundError('Run notebooks 1 and 2 (including their screening cells) before this step.')
classifier, regressor = pd.read_csv(classifier_file), pd.read_csv(regressor_file)
join = ['source', 'canonical_smiles']
for name, df, cols in (('classifier', classifier, join + ['ace_active_probability']),
                       ('regressor', regressor, join + ['predicted_pchembl'])):
    miss = [c for c in cols if c not in df.columns]
    if miss: raise KeyError(f'{name} screen lacks columns {miss}; available: {list(df.columns)}')

def collapse(df, value_cols):
    """One row per (source, canonical_smiles); predictions are deterministic so 'first' is exact."""
    agg = {c: 'first' for c in value_cols if c in df.columns}
    if 'source_id' in df.columns:
        agg['source_id'] = lambda s: ';'.join(sorted(set(s.dropna().astype(str))))
    out = df.groupby(join, as_index=False, sort=False).agg(agg)
    out['n_records'] = df.groupby(join, sort=False).size().to_numpy()
    return out

c = collapse(classifier, ['ace_active_probability', 'max_train_tanimoto']) \
      .rename(columns={'max_train_tanimoto': 'max_train_tanimoto_cls'})
r = collapse(regressor, ['predicted_pchembl', 'predicted_ic50_nM', 'max_train_tanimoto']).rename(columns={'max_train_tanimoto': 'max_train_tanimoto_reg'}).drop(columns=['source_id', 'n_records'], errors='ignore')
candidates = c.merge(r, on=join, how='inner', validate='one_to_one')
print(f'Classifier rows {len(classifier):,} -> {len(c):,} unique; regressor rows {len(regressor):,}; merged {len(candidates):,}')
candidates['model_pass'] = (candidates.ace_active_probability >= CLASSIFIER_CUTOFF) & (candidates.predicted_pchembl >= REGRESSION_CUTOFF_PCHEMBL)
print(f'Model-consensus candidates: {int(candidates.model_pass.sum()):,} / {len(candidates):,}')

Classifier rows 159,658 -> 159,658 unique; regressor rows 159,658; merged 159,658
Model-consensus candidates: 4 / 159,658


In [3]:
def read_any_table(path):
    path = Path(path)
    if path.suffix.lower() in {'.xlsx', '.xls'}: return pd.read_excel(path)
    sample = open(path, encoding='utf-8-sig', newline='').read(30000)
    try: sep = csv.Sniffer().sniff(sample, delimiters=';,\t').delimiter
    except csv.Error: sep = ','
    return pd.read_csv(path, sep=sep, encoding='utf-8-sig', low_memory=False)

def find_col(df, names, required=True):
    norm = lambda x: ''.join(ch for ch in str(x).lower() if ch.isalnum())
    lookup = {norm(col): col for col in df.columns}
    for n in names:
        if norm(n) in lookup: return lookup[norm(n)]
    if required: raise KeyError(f'None of {names} in columns {list(df.columns)}')

alert_path = INPUTS['GLAXO_ALERTS']
assert Path(alert_path).exists(), f'Missing alert table: {alert_path}'
alerts = read_any_table(alert_path)
smarts_col = find_col(alerts, ['smarts', 'smart', 'pattern'])
label_col = find_col(alerts, ['description', 'alert', 'name', 'alert_name'], required=False)
compiled_alerts, invalid = [], []
for ix, row in alerts.iterrows():
    query = Chem.MolFromSmarts(str(row[smarts_col]).strip())
    if query is None: invalid.append((ix, row[smarts_col])); continue
    compiled_alerts.append((str(row[label_col]) if label_col else f'Glaxo rule {ix}', query))
print(f'Compiled Glaxo rules: {len(compiled_alerts)} of {len(alerts)}; invalid SMARTS: {len(invalid)}')
if invalid: print('First invalid rules:', invalid[:5])

def admet_descriptor_row(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {'admet_descriptor_pass': False, 'glaxo_alert_count': np.nan, 'glaxo_alerts': 'invalid_smiles'}
    row = {'mw': Descriptors.MolWt(mol), 'logp': Crippen.MolLogP(mol),
           'hbd': Lipinski.NumHDonors(mol), 'hba': Lipinski.NumHAcceptors(mol),
           'tpsa': rdMolDescriptors.CalcTPSA(mol), 'rotatable_bonds': Lipinski.NumRotatableBonds(mol)}
    row['lipinski_pass'] = bool(row['mw'] <= 500 and row['logp'] <= 5 and row['hbd'] <= 5 and row['hba'] <= 10)
    row['veber_pass'] = bool(row['tpsa'] <= 140 and row['rotatable_bonds'] <= 10)
    hits = [label for label, q in compiled_alerts if mol.HasSubstructMatch(q)]
    row['glaxo_alert_count'], row['glaxo_alerts'] = len(hits), ' | '.join(hits)
    row['admet_descriptor_pass'] = row['lipinski_pass'] and row['veber_pass']
    return row

# Positive control: known ACE inhibitors through the same filters (diagnostic only).
controls = {'captopril': 'C[C@@H](CS)C(=O)N1CCC[C@H]1C(=O)O',
            'enalaprilat': 'C[C@H](N[C@@H](CCc1ccccc1)C(=O)O)C(=O)N1CCC[C@H]1C(=O)O',
            'lisinopril': 'NCCCC[C@H](N[C@@H](CCc1ccccc1)C(=O)O)C(=O)N1CCC[C@H]1C(=O)O'}
control_df = pd.DataFrame({k: admet_descriptor_row(v) for k, v in controls.items()}).T
control_df['glaxo_alert_pass'] = control_df.glaxo_alert_count <= MAX_GLAXO_ALERTS
display(control_df)

Compiled Glaxo rules: 55 of 55; invalid SMARTS: 0


,mw,logp,hbd,hba,tpsa,rotatable_bonds,lipinski_pass,veber_pass,glaxo_alert_count,glaxo_alerts,admet_descriptor_pass,glaxo_alert_pass
captopril,217.29,0.6279,2,3,57.61,3,True,True,1,Thiols,True,False
enalaprilat,348.399,1.1261,3,4,106.94,8,True,True,0,,True,True
lisinopril,405.495,1.2352,4,5,132.96,12,True,False,0,,False,True


In [4]:
# Descriptors/alerts only for model-consensus candidates (the others cannot pass anyway).
mp = candidates[candidates.model_pass]
props = pd.DataFrame(mp.canonical_smiles.map(admet_descriptor_row).tolist(), index=mp.index)
candidates = candidates.join(props)
for col in ('lipinski_pass', 'veber_pass', 'admet_descriptor_pass'):
    candidates[col] = candidates[col].fillna(False).astype(bool)
candidates['glaxo_alert_pass'] = candidates.glaxo_alert_count.le(MAX_GLAXO_ALERTS).fillna(False)
candidates['drugability_pass'] = candidates.model_pass & candidates.admet_descriptor_pass & candidates.glaxo_alert_pass
candidates['priority_score'] = candidates.ace_active_probability * candidates.predicted_pchembl
candidates = candidates.sort_values(['drugability_pass', 'model_pass', 'priority_score'], ascending=[False, False, False])

/tmp/ipykernel_9658/179890247.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  candidates[col] = candidates[col].fillna(False).astype(bool)
/tmp/ipykernel_9658/179890247.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  candidates[col] = candidates[col].fillna(False).astype(bool)
/tmp/ipykernel_9658/179890247.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_si

In [5]:
all_output, pass_output = OUT / 'ace_drugability_all_candidates.csv', OUT / 'ace_druggable_candidates.csv'
candidates.to_csv(all_output, index=False)
passed = candidates[candidates.drugability_pass].copy()
passed.to_csv(pass_output, index=False)
summary = candidates.groupby('source').agg(
    total=('source', 'size'), model_consensus=('model_pass', 'sum'),
    descriptor_pass=('admet_descriptor_pass', 'sum'),
    alerted_among_consensus=('glaxo_alert_count', lambda x: int((x > 0).sum())),
    drugability_pass=('drugability_pass', 'sum')).reset_index()
summary.to_csv(REPORTS / 'drugability_summary_by_source.csv', index=False)
display(summary); display(passed.head(25))
print(f'Wrote {len(passed):,} drugability-passing candidates to {pass_output}')

,source,total,model_consensus,descriptor_pass,alerted_among_consensus,drugability_pass
0,EXPERIMENTAL_ZIAI,7445,0,0,0,0
1,FDA_DRUGBANK,2438,2,1,0,1
2,NPASS,91920,1,0,0,0
3,PLANTS_CMAUP,57855,1,0,0,0


,source,canonical_smiles,ace_active_probability,max_train_tanimoto_cls,source_id,n_records,predicted_pchembl,predicted_ic50_nM,max_train_tanimoto_reg,model_pass,...,tpsa,rotatable_bonds,lipinski_pass,veber_pass,glaxo_alert_count,glaxo_alerts,admet_descriptor_pass,glaxo_alert_pass,drugability_pass,priority_score
1788,FDA_DRUGBANK,CC(NC(CCc1ccccc1)C(=O)O)C(=O)N1CCCC1C(=O)O,0.943056,1.0,Enalaprilat,1,8.42011,3.800935,1.0,True,...,106.94,8.0,True,True,0.0,,True,True,True,7.940637


Wrote 1 drugability-passing candidates to /content/drive/MyDrive/ACE_Workflow/drugability/ace_druggable_candidates.csv


### Interpretation guardrail

A descriptor pass is not a clinical ADMET prediction, and a SMARTS alert is not proof of toxicity. Keep the complete table (including failed candidates and alert labels) for medicinal-chemistry review. Check the positive-control table above: if known ACE inhibitors fail the descriptor or alert filters, those filters are removing real ACE chemotypes and the thresholds should be revisited (with a documented rationale) before interpreting the pass list.